In [0]:
%pip install geopandas gdown keplergl
%restart_python

In [0]:
import os
import geopandas as gpd
import pyspark.databricks.sql.functions as DBF
import pyspark.sql.functions as F

### What ST_ functions are available?

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:schema);

In [0]:
%sql
SHOW FUNCTIONS LIKE 'ST_*'

In [0]:
%sql
SHOW FUNCTIONS LIKE 'H3_*'

### What is the definition of a given ST_ function?

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED st_buffer

### About the Datasets
1. Local Government Areas - 2025 - Shapefile
- Digital boundaries are available in both the Geocentric Datum of Australia 2020 (GDA2020) and the Geocentric Datum of Australia 1994 (GDA94). GDA2020 was adopted as the new official national datum in 2017 and will be adopted gradually by organisations across Australia.

2. Statistical Area Level 3
- Clusters of SA2s representing regions with similar social/economic characteristics.
- Typical Population: 30,000–130,000 people.
- Analyzing regional patterns, such as employment or health.

https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standar[…]3/jul2021-jun2026/access-and-downloads/digital-boundary-files


In [0]:
%skip
import gdown

url = 'https://drive.google.com/file/d/11eAuu-ZrsaxUk_AeWROIw953KiB6_hcW/view?usp=sharing'
uc_volume_path = f'/Volumes/{catalog}/{schema}/{volume}/lga_sa3.zip'

gdown.download(url, uc_volume_path, quiet=False, fuzzy=True)

In [0]:
%skip
import zipfile

extract_path = f'/Volumes/{catalog}/{schema}/{volume}'

In [0]:
%skip
with zipfile.ZipFile(uc_volume_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

- Convert the GeoDataFrame to a Spark DataFrame
- If your geometry column is complex (e.g. Shapely objects), you will need to convert it to WKT or WKB format first since Spark does not natively understand geometry objects.

In [0]:
%skip
import geopandas as gpd
import pandas as pd

gdf_lga = gpd.read_file(f"{extract_path}/lga_sa3/LGA_2025_AUST_GDA2020.shp")
gdf_sa3 = gpd.read_file(f"{extract_path}/lga_sa3/SA3_2021_AUST_GDA2020.shp")


# Convert geometry to WKT
gdf_lga['geometry'] = gdf_lga['geometry'].to_wkt()
gdf_sa3['geometry'] = gdf_sa3['geometry'].to_wkt()


# Convert to Spark DataFrame
lga_pdf = pd.DataFrame(gdf_lga)
lga_sdf = spark.createDataFrame(lga_pdf)

sa3_pdf = pd.DataFrame(gdf_sa3)
sa3_sdf = spark.createDataFrame(sa3_pdf)

# Write to Delta table
lga_sdf.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.lgas")
sa3_sdf.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.sa3")

In [0]:
%skip
# Cluster the lgas and sa3 tables by ste_name21 using SQL
# Assumes catalog and schema are set via widgets

table_lgas = f"{catalog}.{schema}.lgas"
table_sa3 = f"{catalog}.{schema}.sa3"

spark.sql(f"ALTER TABLE {table_lgas} CLUSTER BY (ste_name21)")
spark.sql(f"ALTER TABLE {table_sa3} CLUSTER BY (ste_name21)")

In [0]:
%sql

-- 1. Explore LGA data
SELECT 
    lga_code25 as lga_code,
    lga_name25 as lga_name,
    ste_code21 as state_code,
    ste_name21 as state_name,
    aus_code21 as aus_code,
    aus_name21 as aus_name,
    areasqkm as area_sqkm,
    ST_NPoints(ST_GeomFromText(geometry)) as num_vertices
FROM lgas
WHERE ste_name21 = 'Victoria'
LIMIT 10;

In [0]:
%sql
-- 2. ST_Transform: Area Calculation in Different Coordinate Reference System (CRS)
SELECT 
    lga_code25 as lga_code,
    lga_name25 as lga_name,
    ste_code21 as state_code,
    ste_name21 as state_name,
    areasqkm AS original_area,
    ST_Area(ST_Transform(ST_GeomFromText(geometry, 7844), 3112)) AS area_sqkm_gda2020_zone55, --GDA94 / Geoscience Australia Lambert
    ST_Area(ST_Transform(ST_GeomFromText(geometry, 7844), 3857)) AS area_sqkm_web_mercator --WGS 84 / Pseudo-Mercator -- Spherical Mercator, Google Maps, OpenStreetMap, Bing, ArcGIS, ESRI
FROM lgas
WHERE ste_name21 = 'Victoria'
ORDER BY areasqkm DESC
LIMIT 10;

In [0]:
%sql
-- 3. ST_Contains: Find the LGA that contain a specific point (e.g., Melbourne CBD coordinates)
SELECT 
    lga_code25 as lga_code,
    lga_name25 as lga_name,
    ste_name21 as state_name,
    areasqkm
FROM lgas
WHERE ST_Contains(
    ST_GeomFromText(geometry), 
    ST_Point(144.9631, -37.8136) -- Melbourne CBD coordinates
);

In [0]:
%sql
-- 4. Find which LGA each SA3 is within or intersects (polygon-in-polygon)
SELECT
    s.SA3_CODE21,
    s.SA3_NAME21,
    s.STE_NAME21 as sa3_state,
    s.AREASQKM21 as sa3_area_sqkm,
    l.lga_code25,
    l.lga_name25,
    l.ste_name21 as lga_state,
    l.areasqkm as lga_area_sqkm,
    CASE
        WHEN ST_Within(ST_GeomFromText(s.geometry), ST_GeomFromText(l.geometry)) THEN 'SA3 within LGA'
        WHEN ST_Intersects(ST_GeomFromText(s.geometry), ST_GeomFromText(l.geometry)) THEN 'SA3 intersects with LGA'
        ELSE 'no_relation'
    END as spatial_relation
FROM sa3 s
JOIN lgas l ON ST_Intersects(ST_GeomFromText(s.geometry), ST_GeomFromText(l.geometry))
WHERE l.ste_name21 = 'Victoria'
ORDER BY s.SA3_CODE21 DESC, 
    CASE
        WHEN ST_Within(ST_GeomFromText(s.geometry), ST_GeomFromText(l.geometry)) THEN 0
        ELSE 1
    END
LIMIT 100;

In [0]:
%skip
%sql

-- ============================================================================
-- QUICK START: H3 Indexing Examples
-- ============================================================================

-- H3 POINT-IN-POLYGON: Create H3 indexes for efficient spatial lookups
CREATE OR REPLACE TABLE sa3_h3 AS
SELECT
    *,
    h3_longlatash3(
        ST_X(ST_Centroid(ST_GeomFromText(geometry))),
        ST_Y(ST_Centroid(ST_GeomFromText(geometry))),
        9
    ) as h3_centroid_9
FROM sa3
WHERE ste_name21 = 'Victoria';

CREATE OR REPLACE TABLE lgas_h3 AS
SELECT
    *,
    h3_longlatash3(
        ST_X(ST_Centroid(ST_GeomFromText(geometry))),
        ST_Y(ST_Centroid(ST_GeomFromText(geometry))),
        9
    ) as h3_centroid_9
FROM lgas
WHERE ste_name21 = 'Victoria';

In [0]:
%skip
%sql
CREATE OR REPLACE TABLE sa3_h3_tessellate AS
SELECT
    SA3_CODE21, SA3_NAME21,
    inline(h3_tessellateaswkb(geometry, 9))
FROM sa3
WHERE ste_name21 = 'Victoria';

CREATE OR REPLACE TABLE lgas_h3_tessellate AS
SELECT
    lga_code25, lga_name25,
    inline(h3_tessellateaswkb(geometry, 9)) 
FROM lgas
WHERE ste_name21 = 'Victoria';

In [0]:
%sql
-- 2. H3-BASED SPATIAL JOIN: Use H3 for pre-filtering before spatial operations
SELECT
    s.SA3_CODE21,
    s.SA3_NAME21,
    l.lga_code25,
    l.lga_name25,
    s.h3_centroid_9
FROM sa3_h3 s
JOIN lgas_h3_tessellate l ON s.h3_centroid_9 = l.cellid  -- Fast H3 comparison
WHERE l.core OR ST_Contains(
    st_geomfromwkb(l.chip),
    ST_Centroid(ST_GeomFromText(s.geometry))
)
LIMIT 10;

In [0]:
%sql
-- 3. H3 TESSELLATION: Cover SA3 areas with H3 cells
-- Generate H3 cells that cover each SA3 polygon
SELECT
    SA3_CODE21,
    SA3_NAME21,
    h3_cell,
    h3_resolution(h3_cell) as h3_resolution,
    h3_boundaryaswkt(h3_cell) as h3_boundary
FROM sa3
LATERAL VIEW explode(
    h3_coverash3(
        geometry,
        9  -- H3 resolution 9 (~173km²)
    )
) AS h3_cell
WHERE SA3_NAME21 = 'Ballarat';

In [0]:
# Visualise geometry and H3 columns from _sqldf using kepler.gl

from keplergl import KeplerGl

# Limit rows to avoid large output
pdf = _sqldf.limit(5000).toPandas()

# Try to auto-detect geometry and H3 columns
geometry_col = next((col for col in pdf.columns if col.lower() in ['h3_boundary']), None)
h3_col = next((col for col in pdf.columns if 'h3_cell' in col.lower()), None)

# Prepare KeplerGl config for geometry and H3
config = {
    "version": "v1",
    "config": {
        "visState": {
            "layers": [
                {
                    "id": "geometry_layer",
                    "type": "wkt",
                    "config": {
                        "dataId": "data",
                        "label": "Geometry",
                        "columns": {"geojson": geometry_col},
                        "isVisible": True
                    }
                }
            ] + (
                [{
                    "id": "h3_layer",
                    "type": "h3",
                    "config": {
                        "dataId": "data",
                        "label": "H3",
                        "columns": {"hex_id": h3_col},
                        "isVisible": True
                    }
                }] if h3_col else []
            )
        }
    }
}

map_ = KeplerGl(height=600, data={"data": pdf}, config=config)
display(map_)

In [0]:
%sql
-- 3. H3 TESSELLATION: Cover SA3 areas with H3 cells
-- Generate H3 cells that cover each SA3 polygon
SELECT
    SA3_CODE21,
    SA3_NAME21,
    inline(h3_tessellateaswkb(geometry, 9)),
    st_astext(st_geomfromewkb(chip)) as wkt
FROM sa3
WHERE SA3_NAME21 = 'Ballarat';

In [0]:
# Visualise geometry and H3 columns from _sqldf using kepler.gl

from keplergl import KeplerGl

# Limit rows to avoid large output
pdf = _sqldf.drop('chip').limit(5000).toPandas()

# Try to auto-detect geometry and H3 columns
geometry_col = next((col for col in pdf.columns if col.lower() in ['wkt']), None)
h3_col = next((col for col in pdf.columns if 'cellid' in col.lower()), None)

# Prepare KeplerGl config for geometry and H3
config = {
    "version": "v1",
    "config": {
        "visState": {
            "layers": [
                {
                    "id": "geometry_layer",
                    "type": "wkt",
                    "config": {
                        "dataId": "data",
                        "label": "Geometry",
                        "columns": {"geojson": geometry_col},
                        "isVisible": True
                    }
                }
            ] + (
                [{
                    "id": "h3_layer",
                    "type": "h3",
                    "config": {
                        "dataId": "data",
                        "label": "H3",
                        "columns": {"hex_id": h3_col},
                        "isVisible": True
                    }
                }] if h3_col else []
            )
        }
    }
}

map_ = KeplerGl(height=600, data={"data": pdf}, config=config)
display(map_)

In [0]:
%sql
-- 4. Use H3 distance functions for spatial proximity analysis
SELECT
    s.SA3_CODE21,
    s.SA3_NAME21,
    l.lga_code25,
    l.lga_name25,
    h3_try_distance(s.h3_centroid_9, l.h3_centroid_9) as h3_grid_distance
FROM sa3_h3 s
JOIN lgas_h3 l ON h3_try_distance(s.h3_centroid_9, l.h3_centroid_9) <= 3
WHERE s.STE_NAME21 = 'Victoria'
ORDER BY h3_grid_distance
LIMIT 15;